In [4]:
from __future__ import annotations

import copy
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Set, Tuple
from collections import defaultdict

import numpy as np
import networkx as nx
from joblib import Parallel, delayed
from qiskit import QuantumCircuit, transpile, QuantumRegister
from qiskit.quantum_info import Operator
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import CommutationAnalysis, CommutativeCancellation, Optimize1qGates
from deap import base, creator, tools

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import os

# NEW: 3D clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# =============================
# Utils: plotting helpers
# =============================

def save_plot(name: str, out_dir="out_figs"):
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, f"{name}.png"), dpi=300)


def plot_convergence(hist_eps, save_as: Optional[str] = None):
    plt.figure(); plt.plot(range(len(hist_eps)), hist_eps, marker="o")
    plt.yscale("log"); plt.xlabel("Generation"); plt.ylabel(r"$1\!-\!F$ (log)")
    plt.title("Fidelity convergence"); plt.grid(True); plt.tight_layout()
    if save_as: save_plot(save_as)
    plt.close()


def plot_pareto(front, save_as: Optional[str] = None):
    costs = [i.fitness.values[2] for i in front]
    depths = [i.fitness.values[1] for i in front]
    eps = [1 - i.fitness.values[0] for i in front]
    plt.figure(); sc = plt.scatter(costs, depths, c=eps, cmap="viridis")
    plt.colorbar(sc, label=r"$\varepsilon$ (1-F)")
    plt.xlabel("Chrom. cost"); plt.ylabel("Depth"); plt.title("Local Pareto front")
    plt.gca().invert_yaxis(); plt.tight_layout()
    if save_as: save_plot(save_as)
    plt.close()


def plot_3d_clusters(pareto, n_clusters: int = 4, save_as: Optional[str] = None):
    """3D scatter (Depth, Cost, Error) + K-Means on the Pareto front.

    Axes: X=depth, Y=cost (fitness[2] here = chromosome length), Z=1-F.
    If you want the actual *gate cost* on Y, replace fitness[2] with compute_gate_cost(build(ind)).
    """
    if not pareto:
        return

    data = np.array([
        [ind.fitness.values[1], ind.fitness.values[2], 1.0 - ind.fitness.values[0]]
        for ind in pareto
    ])
    k = max(1, min(n_clusters, len(data)))
    scaler = StandardScaler(); data_scaled = scaler.fit_transform(data)
    kmeans = KMeans(n_clusters=k, n_init=10)
    labels = kmeans.fit_predict(data_scaled)

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(data[:, 0], data[:, 1], data[:, 2], c=labels, cmap='tab10')
    ax.set_xlabel('Depth')
    ax.set_ylabel('Cost')
    ax.set_zlabel('Error ε = 1 - F')
    ax.set_title('3D Clustering (K-Means) — Pareto Population')

    if save_as:
        save_plot(save_as)
        plt.close(fig)
    else:
        plt.tight_layout(); plt.show()


# =============================
# Partition & graph helpers
# =============================

def extract_interblock_gates(qc: QuantumCircuit, blocks: List[Set[int]]) -> List[Tuple]:
    bmap = {q: i for i, bl in enumerate(blocks) for q in bl}
    interblock_gates = []
    for inst, qargs, cargs in qc.data:
        if len(qargs) < 2:
            continue
        qubit_indices = {qc.find_bit(q).index for q in qargs}
        involved_blocks = {bmap.get(q) for q in qubit_indices}
        if len(involved_blocks) > 1:
            interblock_gates.append((inst, qargs, cargs))
    return interblock_gates


# Louvain partition
import community  # python-louvain

def louvain_partition(qc: QuantumCircuit) -> List[Set[int]]:
    """Partitions qubits via Louvain on a weighted interaction graph."""
    G = nx.Graph(); G.add_nodes_from(range(qc.num_qubits))
    for inst, qargs, _ in qc.data:
        if len(qargs) == 2:
            i = qc.find_bit(qargs[0]).index
            j = qc.find_bit(qargs[1]).index
            if G.has_edge(i, j):
                G[i][j]['weight'] += 1
            else:
                G.add_edge(i, j, weight=1)
    partition = community.best_partition(G, weight='weight')
    blocks = defaultdict(set)
    for qubit, community_id in partition.items():
        blocks[community_id].add(qubit)
    return list(blocks.values())


def build_interaction_graph(qc: QuantumCircuit) -> nx.Graph:
    G = nx.Graph(); G.add_nodes_from(range(qc.num_qubits))
    for inst, qargs, _ in qc.data:
        if inst.name in {"cx", "cz", "rzz"} and len(qargs) == 2:
            i, j = [qc.find_bit(q).index for q in qargs]
            w = G.get_edge_data(i, j, default={"weight": 0})["weight"] + 1
            G.add_edge(i, j, weight=w)
    return G


def _partition_metis(graph: nx.Graph) -> List[Set[int]]:
    import nxmetis  # type: ignore
    _, parts = nxmetis.partition(graph, 2)
    return [set(p) for p in parts]


def _partition_kl(graph: nx.Graph) -> List[Set[int]]:
    from networkx.algorithms.community import kernighan_lin_bisection
    a, b = kernighan_lin_bisection(graph); return [set(a), set(b)]


def multilevel_partition(graph: nx.Graph, max_block_size: int) -> List[Set[int]]:
    if len(graph) <= max_block_size:
        return [set(graph.nodes())]
    try:
        parts = _partition_metis(graph)
    except Exception:
        parts = _partition_kl(graph)
    res: List[Set[int]] = []
    for p in parts:
        res.extend(multilevel_partition(graph.subgraph(p), max_block_size))
    return res


def _interblock_gate_cost(qc: QuantumCircuit, blk0: Set[int], blk1: Set[int]) -> int:
    cost = 0
    for inst, qargs, _ in qc.data:
        if len(qargs) < 2:
            continue
        qs = {qc.find_bit(q).index for q in qargs}
        if qs & blk0 and qs & blk1:
            cost += 1
    return cost


def refine_partition_kl(qc: QuantumCircuit, blocks: List[Set[int]], *, max_iter: int = 10) -> List[Set[int]]:
    if len(blocks) < 2:
        return blocks
    a, b = blocks[0].copy(), blocks[1].copy()
    best_cost = _interblock_gate_cost(qc, a, b)
    improved, it = True, 0
    while improved and it < max_iter:
        improved = False; it += 1
        gain_best, q_best, side = 0, None, None
        for q in list(a):
            gain = best_cost - _interblock_gate_cost(qc, a - {q}, b | {q})
            if gain > gain_best:
                gain_best, q_best, side = gain, q, "a2b"
        for q in list(b):
            gain = best_cost - _interblock_gate_cost(qc, a | {q}, b - {q})
            if gain > gain_best:
                gain_best, q_best, side = gain, q, "b2a"
        if gain_best > 0 and q_best is not None:
            improved = True; best_cost -= gain_best
            if side == "a2b":
                a.remove(q_best); b.add(q_best)
            else:
                b.remove(q_best); a.add(q_best)
    blocks[0], blocks[1] = a, b
    return blocks


def extract_commuting_sets(qc: QuantumCircuit):
    try:
        from qiskit.converters import circuit_to_dag
        dag = circuit_to_dag(qc)
        PassManager(CommutationAnalysis()).run(dag)
        layers, seen = [], set()
        for node in dag.topological_op_nodes():
            if node in seen:
                continue
            group = [g for g in node.commutation_set if g.type == "op"]
            layers.append(group); seen.update(group)
        return layers
    except Exception:
        active, lvl, levels = set(), 0, defaultdict(list)
        for inst, qargs, _ in qc.data:
            qub = {qc.find_bit(q).index for q in qargs}
            if active & qub:
                lvl += 1; active.clear()
            active.update(qub); levels[lvl].append((inst, qargs))
        return list(levels.values())


def select_interface_qubits(blocks: List[Set[int]], k: int = 1) -> Dict[int, Set[int]]:
    return {i: set(sorted(bl)[:k]) for i, bl in enumerate(blocks)}


def identify_highly_interactive_qubits(qc: QuantumCircuit, blocks: List[Set[int]], threshold_ratio: float = 0.5) -> Dict[int, int]:
    if len(blocks) < 2:
        return {}
    bmap = {q: i for i, bl in enumerate(blocks) for q in bl}
    qubit_inter_block_counts = defaultdict(lambda: defaultdict(int))
    qubit_total_gate_counts = defaultdict(int)

    for inst, qargs, _ in qc.data:
        q_indices = {qc.find_bit(q).index for q in qargs}
        if len(q_indices) < 2:
            continue
        involved_blocks = {bmap.get(q_idx) for q_idx in q_indices if q_idx in bmap}
        if len(involved_blocks) > 1:
            for q_idx in q_indices:
                if q_idx not in bmap: continue
                qubit_total_gate_counts[q_idx] += 1
                for block_id in involved_blocks:
                    if block_id is not None and bmap[q_idx] != block_id:
                        qubit_inter_block_counts[q_idx][block_id] += 1
        else:
            for q_idx in q_indices:
                if q_idx not in bmap: continue
                qubit_total_gate_counts[q_idx] += 1

    highly_interactive: Dict[int, int] = {}
    for q_idx, inter_counts in qubit_inter_block_counts.items():
        if qubit_total_gate_counts[q_idx] == 0:
            continue
        if inter_counts:
            max_inter_block_id = max(inter_counts, key=inter_counts.get)
            if inter_counts[max_inter_block_id] / qubit_total_gate_counts[q_idx] >= threshold_ratio:
                # Only if not already in the target block
                if q_idx not in blocks[max_inter_block_id]:
                    highly_interactive[q_idx] = max_inter_block_id
    return highly_interactive


def apply_interface_swaps(qc: QuantumCircuit, blocks: List[Set[int]], *,
                          k_interface: int = 1,
                          highly_interactive_qubits: Optional[Dict[int, int]] = None,
                          max_block_size: int = 6) -> QuantumCircuit:
    """Redirects inter-block gates via interface qubits or duplication."""

    def _perform_qubit_duplication(qc: QuantumCircuit, blocks: List[Set[int]],
                                   highly_interactive: Dict[int, int],
                                   max_block_size: int) -> Tuple[Dict[int, int], List[Tuple[str, int, int]]]:
        duplicate_map: Dict[int, int] = {}
        entangle_gates: List[Tuple[str, int, int]] = []
        next_q_idx = qc.num_qubits
        bmap = {q: i for i, bl in enumerate(blocks) for q in bl}
        for q_orig, target_block in highly_interactive.items():
            if bmap.get(q_orig) == target_block:
                continue
            if len(blocks[target_block]) + 1 > max_block_size:
                print(f"⚠️ Block {target_block} full — no duplication for q{q_orig}")
                continue
            q_dup = next_q_idx; next_q_idx += 1
            duplicate_map[q_orig] = q_dup
            blocks[target_block].add(q_dup)
            print(f"🔁 Duplication q{q_orig} → q{q_dup} into block {target_block}")
            entangle_gates.append(('h', q_orig))
            entangle_gates.append(('cx', q_orig, q_dup))
        return duplicate_map, entangle_gates

    iface = select_interface_qubits(blocks, k_interface)
    current_blocks = [s.copy() for s in blocks]
    bmap = {q: i for i, bl in enumerate(current_blocks) for q in bl}

    duplicate_map: Dict[int, int] = {}
    initial_entanglement_gates: List[Tuple[str, int, int]] = []

    if highly_interactive_qubits:
        duplicate_map, initial_entanglement_gates = _perform_qubit_duplication(
            qc, current_blocks, highly_interactive_qubits, max_block_size=max_block_size
        )
        bmap = {q: i for i, bl in enumerate(current_blocks) for q in bl}

    new_num_qubits = max([qc.num_qubits] + list(duplicate_map.values()), default=qc.num_qubits) + 1
    new_qc = QuantumCircuit(new_num_qubits)

    for gate in initial_entanglement_gates:
        if gate[0] == 'h':
            _, q1 = gate; new_qc.h(q1)
        elif gate[0] == 'cx':
            _, q1, q2 = gate; new_qc.cx(q1, q2)

    for inst, qargs, cargs in qc.data:
        mapped_qargs = list(qargs)
        rerouted_by_duplication = False

        for idx, q in enumerate(qargs):
            q_idx = qc.find_bit(q).index
            qubits_in_current_gate_indices = {qc.find_bit(temp_q).index for temp_q in qargs}
            involved_blocks_for_gate = {bmap.get(qi) for qi in qubits_in_current_gate_indices if qi in bmap}
            if len(involved_blocks_for_gate) > 1:
                if q_idx in duplicate_map and bmap.get(duplicate_map[q_idx]) in involved_blocks_for_gate:
                    mapped_qargs[idx] = new_qc.qubits[duplicate_map[q_idx]]
                    rerouted_by_duplication = True

        final_gate_q_indices = {q_obj.index for q_obj in mapped_qargs}
        final_involved_blocks = {bmap.get(q_idx) for q_idx in final_gate_q_indices if q_idx in bmap}

        if len(final_involved_blocks) <= 1 and not rerouted_by_duplication:
            new_qc.append(inst, [new_qc.qubits[qc.find_bit(q).index] for q in qargs], cargs)
            continue

        pre_swaps, post_swaps = [], []
        if not rerouted_by_duplication:
            for idx, q in enumerate(qargs):
                q_idx = qc.find_bit(q).index
                blk_of_q = bmap.get(q_idx)
                if blk_of_q is None:
                    continue
                if q_idx not in iface[blk_of_q]:
                    target_iface_q = next(iter(iface[blk_of_q]), None)
                    if target_iface_q is not None:
                        current_q_indices_in_mapped = {q_obj.index for q_obj in mapped_qargs}
                        if target_iface_q not in current_q_indices_in_mapped:
                            pre_swaps.append((q_idx, target_iface_q))
                            post_swaps.append((q_idx, target_iface_q))
                            mapped_qargs[idx] = new_qc.qubits[target_iface_q]

        for a, b in pre_swaps:
            new_qc.swap(a, b)
        final_gate_qargs_for_append = [new_qc.qubits[q_obj.index] for q_obj in mapped_qargs]
        new_qc.append(inst, final_gate_qargs_for_append, cargs)
        for a, b in reversed(post_swaps):
            new_qc.swap(a, b)

    return new_qc


# =============================
# Subcircuits & recomposition
# =============================

def extract_subcircuit(qc: QuantumCircuit, qubits: Set[int]) -> QuantumCircuit:
    sub = QuantumCircuit(len(qubits))
    idx = {q: i for i, q in enumerate(sorted(list(qubits)))}
    for inst, qargs, cargs in qc.data:
        if all(qc.find_bit(q).index in qubits for q in qargs):
            remap = [sub.qubits[idx[qc.find_bit(q).index]] for q in qargs]
            sub.append(inst, remap, cargs)
    return sub


def recompose_from_blocks(qc_original: QuantumCircuit,
                          block_subcircuits: List[Tuple[Set[int], QuantumCircuit]]) -> QuantumCircuit:
    block_maps = []
    for block_qubits, sub in block_subcircuits:
        sorted_block = sorted(block_qubits)
        local_to_global = {i: q for i, q in enumerate(sorted_block)}
        global_to_local = {q: i for i, q in enumerate(sorted_block)}
        block_maps.append((set(sorted_block), sub, global_to_local))

    num_qubits = qc_original.num_qubits
    qc_recomposed = QuantumCircuit(num_qubits)
    subcircuit_cursors = [0 for _ in block_subcircuits]

    for inst, qargs, cargs in qc_original.data:
        q_indices = [qc_original.find_bit(q).index for q in qargs]
        inserted = False
        for idx, (block_qubits, sub, g2l) in enumerate(block_maps):
            if all(q in block_qubits for q in q_indices):
                if subcircuit_cursors[idx] >= len(sub.data):
                    raise ValueError(f"Too many gates in block {idx} of the original circuit.")
                inst_opt, qargs_opt, cargs_opt = sub.data[subcircuit_cursors[idx]]
                subcircuit_cursors[idx] += 1
                mapped_qargs = [
                    qc_recomposed.qubits[sorted(block_qubits)[sub.find_bit(q).index]] for q in qargs_opt
                ]
                qc_recomposed.append(inst_opt, mapped_qargs, cargs_opt)
                inserted = True
                break
        if not inserted:
            mapped_qargs = [qc_recomposed.qubits[i] for i in q_indices]
            qc_recomposed.append(inst, mapped_qargs, cargs)
    return qc_recomposed


# =============================
# Fidelity & compression utils
# =============================

def compute_fidelity(circ: QuantumCircuit, target: np.ndarray) -> float:
    circ_op = Operator(circ).data
    target_nqubits = int(np.log2(target.shape[0]))
    if circ.num_qubits > target_nqubits:
        target_op_padded = np.eye(2**circ.num_qubits, dtype=complex)
        target_op_padded[:target.shape[0], :target.shape[1]] = target
        target = target_op_padded
    elif circ.num_qubits < target_nqubits:
        raise ValueError("Circuit has fewer qubits than target operator, cannot compute fidelity directly.")
    return abs(np.trace(circ_op @ target.conj().T)) / (2 ** circ.num_qubits)


def cancel_inverse_gates(c: QuantumCircuit) -> QuantumCircuit:
    new, skip = QuantumCircuit(c.num_qubits), set()
    for i in range(len(c.data) - 1):
        if i in skip:
            continue
        g1, q1, _ = c.data[i]; g2, q2, _ = c.data[i + 1]
        if g1.name == g2.name and q1 == q2 and g1.name in {"x", "y", "z", "h", "cx"}:
            skip.add(i + 1); continue
        new.append(g1, q1)
    if (len(c.data) - 1) not in skip:
        g, q, _ = c.data[-1]; new.append(g, q)
    return new


def merge_rotations(c: QuantumCircuit) -> QuantumCircuit:
    new, i = QuantumCircuit(c.num_qubits), 0
    while i < len(c.data):
        g, q, _ = c.data[i]
        if g.name in {"rx", "ry", "rz"}:
            angle, j = g.params[0], i + 1
            while j < len(c.data):
                g2, q2, _ = c.data[j]
                if g2.name == g.name and q2 == q:
                    angle += g2.params[0]; j += 1
                else:
                    break
            getattr(new, g.name)(angle, q[0]); i = j
        else:
            new.append(g, q); i += 1
    return new


def remove_negligible_rotations(c: QuantumCircuit, *, th: float = 1e-4) -> QuantumCircuit:
    new = QuantumCircuit(c.num_qubits)
    for g, q, _ in c.data:
        if g.name in {"rx", "ry", "rz"} and abs(g.params[0]) < th:
            continue
        new.append(g, q)
    return new


def compress_custom(circ: QuantumCircuit) -> QuantumCircuit:
    return remove_negligible_rotations(merge_rotations(cancel_inverse_gates(circ)))


def qiskit_opt_pass(c: QuantumCircuit) -> QuantumCircuit:
    return PassManager([Optimize1qGates(), CommutativeCancellation()]).run(c)


# =============================
# NSGA-II block optimizer (+ Local Angle Search)
# =============================

def compute_gate_cost(qc: QuantumCircuit) -> float:
    cost_table = {"x": 1, "z": 1, "s": 1, "sdg": 1, "t": 1, "tdg": 1,
                  "h": 2, "cx": 5, "cz": 5, "ccx": 13}
    return sum(cost_table.get(inst.name.lower(), 1) for inst, _, _ in qc.data)


def update_rotation_angles(
    chrom: List[Tuple[str, int, Optional[int], Optional[float]]],
    build_fn,
    target_unitary: np.ndarray,
    *,
    eta_range: Sequence[float] = (0.01, 0.1, 0.5),
    delta: float = 0.1,
) -> List[Tuple[str, int, Optional[int], Optional[float]]]:
    """
    Local search via central differences on the rotation parameters.
    Compatible genes: (name, target, ctrl, theta), name in {'rx','ry','rz','rzz'} (lowercase).
    """
    def wrap_angle(theta: float) -> float:
        twopi = 2.0 * math.pi
        return ((theta + math.pi) % twopi) - math.pi

    def set_angle(ch: List[Tuple[str, int, Optional[int], Optional[float]]], idx: int, theta: float):
        g, t, c, a = ch[idx]
        ch2 = list(ch)
        ch2[idx] = (g, t, c, wrap_angle(theta))
        return ch2

    def fitness_of(ch: List[Tuple[str, int, Optional[int], Optional[float]]]) -> float:
        qc = build_fn(ch)
        return compute_fidelity(qc, target_unitary)

    best = list(chrom)
    base_fit = fitness_of(best)

    for i, gene in enumerate(best):
        g, t, c, a = gene
        if g.lower() not in {"rx", "ry", "rz", "rzz"} or a is None:
            continue

        theta0 = float(a)
        f_plus  = fitness_of(set_angle(best, i, theta0 + delta))
        f_minus = fitness_of(set_angle(best, i, theta0 - delta))
        grad = (f_plus - f_minus) / (2.0 * delta)

        best_local_fit = base_fit
        for eta in eta_range:
            cand_theta = theta0 + eta * grad
            cand = set_angle(best, i, cand_theta)
            f = fitness_of(cand)
            if f > best_local_fit:
                best_local_fit = f
                best = cand  # immediate acceptance
        base_fit = best_local_fit

    return best


'''def optimise_block_nsga2(
    qc_target: QuantumCircuit,
    *,
    generations: int = 500,
    pop_size: int = 300,
    n_jobs: int = -1,
    las_enable: bool = True,
    las_eta_range: Sequence[float] = (0.01, 0.1, 0.5),
    las_delta: float = 0.1,
):
    """
    Intra-block NSGA-II with integrated local angle search (LAS).
    """
    nq = qc_target.num_qubits; U_target = Operator(qc_target).data
    gate_pool = ["h", "x", "y", "z", "rx", "ry", "rz", "cx", "cz", "rzz"]

    def gen_gene():
        g = random.choice(gate_pool); tgt = random.randrange(nq)
        if g in {"rx", "ry", "rz"}:
            return (g, tgt, None, random.uniform(0, 2 * math.pi))
        if g == "rzz":
            ctrl = random.choice([q for q in range(nq) if q != tgt])
            return (g, tgt, ctrl, random.uniform(0, 2 * math.pi))
        if g in {"cx", "cz"}:
            ctrl = random.choice([q for q in range(nq) if q != tgt])
            return (g, tgt, ctrl, None)
        return (g, tgt, None, None)

    def build(ch):
        qc = QuantumCircuit(nq)
        for g, t, ctrl, a in ch:
            if g == "rzz":
                qc.rzz(a, ctrl, t)
            elif g in {"cx", "cz"}:
                getattr(qc, g)(ctrl, t)
            elif g in {"rx", "ry", "rz"}:
                getattr(qc, g)(a, t)
            else:
                getattr(qc, g)(t)
        return qc

    def eval_ind(ind):
        qc = build(ind); fid = compute_fidelity(qc, U_target)
        depth = transpile(qc, basis_gates=["cx", "rz", "sx"], optimization_level=1).depth()
        cost = len(ind)  # fast proxy; replaceable by compute_gate_cost(qc)
        return fid, depth, cost

    if not hasattr(creator, "FitnessMulti"):
        creator.create("FitnessMulti", base.Fitness, weights=(1, -1, -1))
        creator.create("Individual", list, fitness=creator.FitnessMulti)
    tb = base.Toolbox(); tb.register("gene", gen_gene)
    tb.register("individual", tools.initRepeat, creator.Individual, tb.gene, 15)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("mate", tools.cxTwoPoint)

    def mut_replace(ind):
        ind[random.randrange(len(ind))] = gen_gene()
        return (ind,)

    tb.register("mutate", mut_replace)
    tb.register("select", tools.selNSGA2)

    pop = tb.population(pop_size)
    fits = Parallel(n_jobs)(delayed(eval_ind)(i) for i in pop)
    for ind, fit in zip(pop, fits):
        ind.fitness.values = fit
    hist_eps = [1 - max(pop, key=lambda i: i.fitness.values[0]).fitness.values[0]]

    for gen in range(generations):
        tools.emo.assignCrowdingDist(pop)
        offspring = tools.selTournamentDCD(pop, len(pop)); offspring = list(map(tb.clone, offspring))
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < 0.9:
                tb.mate(c1, c2)
                if hasattr(c1.fitness, "values"): del c1.fitness.values
                if hasattr(c2.fitness, "values"): del c2.fitness.values
        for ind in offspring:
            if random.random() < 0.9:
                tb.mutate(ind)
                if hasattr(ind.fitness, "values"): del ind.fitness.values

        invalid = [i for i in offspring if not i.fitness.valid]
        fits = Parallel(n_jobs)(delayed(eval_ind)(i) for i in invalid)
        for ind, fit in zip(invalid, fits):
            ind.fitness.values = fit

        # ===== Local angle search (LAS) =====
        if las_enable:
            improved: List[creator.Individual] = []
            for ind in offspring:
                tuned = update_rotation_angles(
                    list(ind), build, U_target,
                    eta_range=las_eta_range, delta=las_delta
                )
                if tuned != list(ind):
                    new_ind = creator.Individual(tuned)
                    new_ind.fitness.values = eval_ind(new_ind)
                    if new_ind.fitness.values[0] > ind.fitness.values[0]:
                        improved.append(new_ind)
            offspring.extend(improved)
        # =============================================

        pop = tb.select(pop + offspring, k=len(pop))
        best = max(pop, key=lambda i: i.fitness.values[0])
        hist_eps.append(1 - best.fitness.values[0])
        print(f"Gen {gen + 1:>4} | Fid {best.fitness.values[0]:.4f} | "
              f"D {best.fitness.values[1]:>3} | C {best.fitness.values[2]:>3}")

    front = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    plot_convergence(hist_eps, save_as=f"block_fid_conv_{nq}q")
    plot_pareto(front, save_as=f"block_pareto_{nq}q")
    plot_3d_clusters(front, n_clusters=4, save_as=f"block_clusters3d_{nq}q")

    return build(max(pop, key=lambda i: i.fitness.values[0]))

'''
def optimise_block_nsga2(
    qc_target: QuantumCircuit,
    *,
    generations: int = 500,
    pop_size: int = 300,
    n_jobs: int = -1,
    las_enable: bool = True,
    las_eta_range: Sequence[float] = (0.01, 0.1, 0.5),
    las_delta: float = 0.1,
    las_top_k: int = 10,  # NEW: n best individuals for the local search
):
    """
    Intra-block NSGA-II with optional local angle search (LAS).
    If `las_enable` is True, LAS is applied **only** to the `las_top_k`
    best individuals (by fidelity) at each generation.
    """
    nq = qc_target.num_qubits
    U_target = Operator(qc_target).data
    gate_pool = ["h", "x", "y", "z", "rx", "ry", "rz", "cx", "cz", "rzz"]

    def gen_gene():
        g = random.choice(gate_pool)
        tgt = random.randrange(nq)
        if g in {"rx", "ry", "rz"}:
            return (g, tgt, None, random.uniform(0, 2 * math.pi))
        if g == "rzz":
            ctrl = random.choice([q for q in range(nq) if q != tgt])
            return (g, tgt, ctrl, random.uniform(0, 2 * math.pi))
        if g in {"cx", "cz"}:
            ctrl = random.choice([q for q in range(nq) if q != tgt])
            return (g, tgt, ctrl, None)
        return (g, tgt, None, None)

    def build(ch):
        qc = QuantumCircuit(nq)
        for g, t, ctrl, a in ch:
            if g == "rzz":
                qc.rzz(a, ctrl, t)
            elif g in {"cx", "cz"}:
                getattr(qc, g)(ctrl, t)
            elif g in {"rx", "ry", "rz"}:
                getattr(qc, g)(a, t)
            else:
                getattr(qc, g)(t)
        return qc

    def eval_ind(ind):
        qc = build(ind)
        fid = compute_fidelity(qc, U_target)
        depth = transpile(qc, basis_gates=["cx", "rz", "sx"], optimization_level=1).depth()
        cost = len(ind)  # fast proxy; replaceable by compute_gate_cost(qc)
        return fid, depth, cost

    # === DEAP setup ===
    if not hasattr(creator, "FitnessMulti"):
        creator.create("FitnessMulti", base.Fitness, weights=(1, -1, -1))
        creator.create("Individual", list, fitness=creator.FitnessMulti)
    tb = base.Toolbox()
    tb.register("gene", gen_gene)
    tb.register("individual", tools.initRepeat, creator.Individual, tb.gene, 15)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("mate", tools.cxTwoPoint)

    def mut_replace(ind):
        ind[random.randrange(len(ind))] = gen_gene()
        return (ind,)

    tb.register("mutate", mut_replace)
    tb.register("select", tools.selNSGA2)

    # === Init population ===
    pop = tb.population(pop_size)
    fits = Parallel(n_jobs)(delayed(eval_ind)(i) for i in pop)
    for ind, fit in zip(pop, fits):
        ind.fitness.values = fit
    hist_eps = [1 - max(pop, key=lambda i: i.fitness.values[0]).fitness.values[0]]

    # === Generational loop ===
    for gen in range(generations):
        tools.emo.assignCrowdingDist(pop)
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = list(map(tb.clone, offspring))

        # Crossover
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < 0.9:
                tb.mate(c1, c2)
                if hasattr(c1.fitness, "values"):
                    del c1.fitness.values
                if hasattr(c2.fitness, "values"):
                    del c2.fitness.values

        # Mutation
        for ind in offspring:
            if random.random() < 0.9:
                tb.mutate(ind)
                if hasattr(ind.fitness, "values"):
                    del ind.fitness.values

        # Evaluate only the invalid individuals
        invalid = [i for i in offspring if not i.fitness.valid]
        if invalid:
            fits = Parallel(n_jobs)(delayed(eval_ind)(i) for i in invalid)
            for ind, fit in zip(invalid, fits):
                ind.fitness.values = fit

        # ===== Local angle search (LAS) on top-k =====
        if las_enable and las_top_k > 0:
            # Sort the offspring by decreasing fidelity
            sorted_offspring = sorted(offspring, key=lambda i: i.fitness.values[0], reverse=True)
            k = min(las_top_k, len(sorted_offspring))
            candidates = sorted_offspring[:k]

            improved = []
            for ind in candidates:
                tuned = update_rotation_angles(
                    list(ind), build, U_target,
                    eta_range=las_eta_range, delta=las_delta
                )
                if tuned != list(ind):
                    new_ind = creator.Individual(tuned)
                    new_ind.fitness.values = eval_ind(new_ind)
                    if new_ind.fitness.values[0] > ind.fitness.values[0]:
                        improved.append(new_ind)
            # Only add the improved ones (moderate number)
            if improved:
                offspring.extend(improved)
        # =============================================

        # NSGA-II environmental selection
        pop = tb.select(pop + offspring, k=len(pop))

        best = max(pop, key=lambda i: i.fitness.values[0])
        hist_eps.append(1 - best.fitness.values[0])
        print(
            f"Gen {gen + 1:>4} | Fid {best.fitness.values[0]:.4f} | "
            f"D {best.fitness.values[1]:>3} | C {best.fitness.values[2]:>3}"
            + (f" | LAS@Top{k if las_enable else 0}" if las_enable else "")
        )

    # Pareto front + visualizations
    front = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    plot_convergence(hist_eps, save_as=f"block_fid_conv_{nq}q")
    plot_pareto(front, save_as=f"block_pareto_{nq}q")
    plot_3d_clusters(front, n_clusters=4, save_as=f"block_clusters3d_{nq}q")

    # Return the individual with the best fidelity
    return build(max(pop, key=lambda i: i.fitness.values[0]))

# =============================
# Inter-block injection (SA / stochastic)
# =============================

@dataclass
class InjectionGate:
    gate: str
    q1: int
    q2: int
    theta: Optional[float]
    enabled: bool = True

    def copy(self) -> "InjectionGate":
        return InjectionGate(self.gate, self.q1, self.q2, self.theta, self.enabled)


def _sa_build_circuit(base: QuantumCircuit, injections: Sequence[InjectionGate]) -> QuantumCircuit:
    circ = base.copy()
    for inj in injections:
        if not inj.enabled:
            continue
        if inj.gate == "rzz":
            circ.rzz(inj.theta, inj.q1, inj.q2)
        else:
            getattr(circ, inj.gate)(inj.q1, inj.q2)
    circ = transpile(circ, basis_gates=["cx", "rz", "sx"], optimization_level=1)
    return circ


def _sa_energy(injections: Sequence[InjectionGate], *, base: QuantumCircuit, target_U: np.ndarray,
               α: float, β: float, γ: float, δ: float, fid_tol: float,
               crosstalk_mat: Optional[np.ndarray]) -> float:
    cand = _sa_build_circuit(base, injections)
    n2q = sum(1 for inj in injections if inj.enabled)
    depth = cand.depth() or 0
    crosstalk = 0.0
    if crosstalk_mat is not None:
        for inj in injections:
            if inj.enabled:
                crosstalk += crosstalk_mat[inj.q1, inj.q2]
    fid = compute_fidelity(cand, target_U)
    fid_penalty = (1.0 - fid) / fid_tol
    return α * n2q + β * depth + γ * crosstalk + δ * fid_penalty


def _sa_rand_move(injections: Sequence[InjectionGate], blocks: List[Set[int]], *, rng: random.Random,
                   eps_theta: float = 0.1) -> List[InjectionGate]:
    moves = ["toggle", "swap_type", "shift", "tune_theta"]
    choice = rng.choice(moves)
    cand = [inj.copy() for inj in injections]
    idx = rng.randrange(len(cand))
    inj = cand[idx]
    if choice == "toggle":
        inj.enabled = not inj.enabled
    elif choice == "swap_type":
        inj.gate = rng.choice([g for g in ("cx", "cz", "rzz") if g != inj.gate])
        inj.theta = None if inj.gate != "rzz" else rng.uniform(0, 2 * math.pi)
    elif choice == "shift":
        blk0, blk1 = blocks[0], blocks[1]
        inj.q1 = rng.choice(tuple(blk0)); inj.q2 = rng.choice(tuple(blk1))
    elif choice == "tune_theta" and inj.gate == "rzz":
        inj.theta = (inj.theta or 0.0) + rng.uniform(-eps_theta, eps_theta)
    return cand


def _sa_generate_pool(blocks: List[Set[int]], gate_types: Sequence[str], *, rng: random.Random,
                      n_candidates: int) -> List[InjectionGate]:
    blk0, blk1 = blocks[0], blocks[1]; pool: List[InjectionGate] = []
    for _ in range(n_candidates):
        gate = rng.choice(gate_types)
        q1 = rng.choice(tuple(blk0)); q2 = rng.choice(tuple(blk1))
        theta = rng.uniform(0, 2 * math.pi) if gate == "rzz" else None
        pool.append(InjectionGate(gate, q1, q2, theta, enabled=False))
    return pool


def sa_injection(base_qc: QuantumCircuit, blocks: List[Set[int]], *,
                 gate_types: Sequence[str] = ("cx", "cz", "rzz"),
                 n_candidates: int = 120,
                 fid_threshold: float = 0.999,
                 n_iters: int = 2000,
                 α: float = 1.0, β: float = 0.01, γ: float = 0.0, δ: float = 1e4,
                 schedule_alpha: float = 0.85,
                 seed: Optional[int] = None,
                 crosstalk_mat: Optional[np.ndarray] = None) -> Tuple[QuantumCircuit, List[Tuple[str, int, int, Optional[float]]]]:
    if len(blocks) < 2:
        raise ValueError("sa_injection requires at least two blocks.")
    rng = random.Random(seed)
    injections = _sa_generate_pool(blocks, gate_types, rng=rng, n_candidates=n_candidates)
    target_U = Operator(base_qc).data
    sample_E = []
    for _ in range(30):
        tmp = _sa_rand_move(injections, blocks, rng=rng)
        e = _sa_energy(tmp, base=base_qc, target_U=target_U, α=α, β=β, γ=γ, δ=δ,
                         fid_tol=1.0 - fid_threshold, crosstalk_mat=crosstalk_mat)
        sample_E.append(e)
    T = 5.0 * (np.std(sample_E) or 1.0)
    best = copy.deepcopy(injections)
    E_best = _sa_energy(best, base=base_qc, target_U=target_U, α=α, β=β, γ=γ, δ=δ,
                         fid_tol=1.0 - fid_threshold, crosstalk_mat=crosstalk_mat)
    current, E_curr = copy.deepcopy(best), E_best

    for _ in range(n_iters):
        cand = _sa_rand_move(current, blocks, rng=rng)
        E_cand = _sa_energy(cand, base=base_qc, target_U=target_U, α=α, β=β, γ=γ, δ=δ,
                              fid_tol=1.0 - fid_threshold, crosstalk_mat=crosstalk_mat)
        ΔE = E_cand - E_curr
        accept = ΔE < 0 or rng.random() < math.exp(-ΔE / T)
        if accept:
            current, E_curr = cand, E_cand
            if E_curr < E_best:
                best, E_best = copy.deepcopy(current), E_curr
        T *= schedule_alpha

    final_circ = _sa_build_circuit(base_qc, best)
    fid_final = compute_fidelity(final_circ, target_U)
    if fid_final < fid_threshold:
        raise RuntimeError(f"SA does not reach the target fidelity: {fid_final:.5f} < {fid_threshold}")

    kept = [(inj.gate, inj.q1, inj.q2, inj.theta) for inj in best if inj.enabled]
    return final_circ, kept


'''def stochastic_injection(qc: QuantumCircuit, blocks: List[Set[int]], *,
                         n_injections: int = 100,
                         fid_threshold: float = 0.999,
                         gate_probs: Optional[Dict[str, float]] = None) -> Tuple[QuantumCircuit, List[Tuple[str, int, int, Optional[float]]]]:
    if len(blocks) < 2:
        raise ValueError("stochastic_injection requires at least two blocks.")
    gate_probs = gate_probs or {"cx": 1.0, "cz": 1.0, "rzz": 1.0}
    total = sum(gate_probs.values())
    gate_types, probs = zip(*([(g, p / total) for g, p in gate_probs.items()]))

    rng = random.Random()
    kept: List[Tuple[str, int, int, Optional[float]]] = []
    U_ref = Operator(qc).data

    for _ in range(n_injections):
        gate = rng.choices(gate_types, probs, k=1)[0]
        qi = rng.choice(tuple(blocks[0])); qj = rng.choice(tuple(blocks[1]))
        cand = qc.copy()
        if gate == "rzz":
            theta = rng.uniform(0, 2 * math.pi); cand.rzz(theta, qi, qj)
        else:
            theta = None; getattr(cand, gate)(qi, qj)
        cand = qiskit_opt_pass(compress_custom(cand))
        fid = compute_fidelity(cand, U_ref)
        if fid >= fid_threshold:
            qc = cand; kept.append((gate, qi, qj, theta)); U_ref = Operator(qc).data
    return qc, kept'''

def stochastic_injection(qc: QuantumCircuit, blocks: List[Set[int]], *,
                         n_injections: int = 100,
                         fid_threshold: float = 0.999,
                         gate_probs: Optional[Dict[str, float]] = None
                         ) -> Tuple[QuantumCircuit, List[Tuple[str, int, int, Optional[float]]]]:
    # --- Preliminary checks ---
    if len(blocks) < 2:
        raise ValueError("stochastic_injection requires at least two blocks.")
    if any(len(b) == 0 for b in blocks):
        raise ValueError("All blocks must be non-empty.")

    gate_probs = gate_probs or {"cx": 1.0, "cz": 1.0, "rzz": 1.0}
    total = float(sum(gate_probs.values()))
    if total <= 0:
        raise ValueError("Gate probabilities must be strictly positive.")

    gate_types, probs = zip(*((g, p / total) for g, p in gate_probs.items()))

    rng = random.Random()
    kept: List[Tuple[str, int, int, Optional[float]]] = []
    U_ref = Operator(qc).data

    # --- Injection loop ---
    for _ in range(n_injections):
        gate = rng.choices(gate_types, probs, k=1)[0]

        # Choose two distinct, non-empty blocks
        # (in case the list contains >2 blocks)
        max_tries = 50
        for _try in range(max_tries):
            i = rng.randrange(len(blocks))
            j = rng.randrange(len(blocks))
            if i != j and len(blocks[i]) > 0 and len(blocks[j]) > 0:
                # Choose two qubits (potentially the same values if overlapping)
                qi = rng.choice(tuple(blocks[i]))
                qj = rng.choice(tuple(blocks[j]))
                # Make sure these are NOT the same indices
                if qi != qj:
                    break
        else:
            raise ValueError(
                "Unable to select two distinct qubits among the provided blocks. "
                "Check that the blocks do not reduce to the same qubit."
            )

        # Build the candidate
        cand = qc.copy()
        if gate == "rzz":
            theta = rng.uniform(0, 2 * math.pi)
            cand.rzz(theta, qi, qj)
        else:
            theta = None
            # cx/cz etc. require two distinct qubits (already guaranteed)
            getattr(cand, gate)(qi, qj)

        # Possible optimization/compression (assumed defined in your code)
        cand = qiskit_opt_pass(compress_custom(cand))

        # Fidelity computation (assumed defined)
        fid = compute_fidelity(cand, U_ref)

        if fid >= fid_threshold:
            qc = cand
            kept.append((gate, qi, qj, theta))
            U_ref = Operator(qc).data

    return qc, kept





# =============================
# Fidelity-driven injection (greedy)
# =============================

def fidelity_driven_injection(
    base_qc: QuantumCircuit,
    target_qc: QuantumCircuit,
    blocks: List[Set[int]],
    max_trials: int = 300,
    fid_threshold: float = 0.9999,
) -> Tuple[QuantumCircuit, List[Tuple[str, int, int, Optional[float]]]]:
    target_unitary = Operator(target_qc).data
    candidate_qc = base_qc.copy()
    kept_injections: List[Tuple[str, int, int, Optional[float]]] = []

    gate_pool = ["cx", "cz", "rzz"]
    rng = random.Random(42)

    for _ in range(max_trials):
        gate = rng.choice(gate_pool)
        q1 = rng.choice(tuple(blocks[0]))
        q2 = rng.choice(tuple(blocks[1]))
        theta = rng.uniform(0, 2 * math.pi) if gate == "rzz" else None

        test_qc = candidate_qc.copy()
        if gate == "rzz":
            test_qc.rzz(theta, q1, q2)
        else:
            getattr(test_qc, gate)(q1, q2)

        fid = compute_fidelity(test_qc, target_unitary)
        if fid > compute_fidelity(candidate_qc, target_unitary):
            candidate_qc = test_qc
            kept_injections.append((gate, q1, q2, theta))
            print(f"✅ Added {gate}({q1},{q2}) [fid={fid:.5f}]")
            if fid >= fid_threshold:
                break
        else:
            print(f"❌ Rejected {gate}({q1},{q2}) [fid={fid:.5f}]")

    return candidate_qc, kept_injections


# =============================
# Full pipeline
# =============================

def optimise_circuit_pipeline(
    qc: QuantumCircuit,
    *,
    max_block_size: int = 5,
    k_interface: int = 1,
    injection_method: str = "stochastic",  # "sa" ou "stochastic"
    fid_threshold: float = 0.999,
    sa_iters: int = 2500,
    sa_seed: Optional[int] = 42,
    qubit_duplication_threshold: float = 0.5,
) -> Tuple[QuantumCircuit, Dict[str, object]]:

    print("\nOriginal circuit:")
    print(qc.draw(output="text"))
    qc.draw('mpl', filename='circuit_original.png', style='mpl', fold=1)

    qc_orig = qc.copy()
    U_orig = Operator(qc_orig).data
    cost_orig = compute_gate_cost(qc_orig)
    print(f"💰 Cost of the original circuit (Lee et al. 2006): {cost_orig}")

    print("\n📌 Partitioning the initial circuit…")
    G = build_interaction_graph(qc)
    #original_blocks = louvain_partition(qc)
    original_blocks = multilevel_partition(G,max_block_size)
    print("Qubits per block (initial):", tuple(original_blocks))

    original_interblock_gates = extract_interblock_gates(qc, original_blocks)
    print(f"📎 {len(original_interblock_gates)} inter-block gates extracted for later reinjection.")

    print("🧭 Displaying the interaction graph… before duplication")
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    edge_weights = nx.get_edge_attributes(G, 'weight')
    nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=800, font_size=12, font_weight='bold')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weights, font_color='red')
    plt.title("Interaction graph before duplication")
    plt.tight_layout(); save_plot("interaction_graph_avant_duplication"); plt.close()

    highly_interactive_qubits = identify_highly_interactive_qubits(qc, original_blocks, qubit_duplication_threshold)
    if highly_interactive_qubits:
        print("💡 Qubits identified for duplication (original_q: target_block):", highly_interactive_qubits)
    else:
        print("💡 No qubit duplication necessary or identified.")

    for orig_q, target_block in highly_interactive_qubits.items():
        original_blocks[target_block].add(orig_q)
        print(f"🧪 Qubit {orig_q} added to block {target_block} for NSGA-II")

    print("🧭 Displaying the interaction graph…")
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    edge_weights = nx.get_edge_attributes(G, 'weight')
    nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=800, font_size=12, font_weight='bold')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weights, font_color='red')
    plt.title("Interaction graph")
    plt.tight_layout(); save_plot("interaction_graph"); plt.close()

    # 2. Intra-block optimization
    block_circuits: List[Tuple[List[int], QuantumCircuit]] = []
    for idx, bl in enumerate(original_blocks):
        sub = extract_subcircuit(qc, bl)
        print(f"\n––– Block {idx} | Qubits {sorted(bl)} –––")
        print(sub.draw(output="text"))
        sub.draw('mpl', filename=f"block_{idx}_circuit_original.png", style='mpl', fold=1)
        print("  → NSGA-II optimization in progress…")
        best = optimise_block_nsga2(sub, generations=500, pop_size=400)

        from qiskit.visualization import circuit_drawer
        fig = circuit_drawer(best, output="mpl", fold=60, style={"fontsize": 12})
        os.makedirs("out_figs", exist_ok=True)
        fig.savefig(f"out_figs/block_{idx}_circuit_optimized.png", dpi=300, bbox_inches='tight')
        plt.close(fig)
        print("    ✅ Optimized circuit:")
        print(best.draw(output="text"))
        block_circuits.append((sorted(list(bl)), best))
        best.draw('mpl', filename=f"optimized_block_{idx}_circuit.png", style='mpl', fold=1)

    qc_rebuilt_original_qubits = QuantumCircuit(qc.num_qubits)
    for qubits_list, cir in block_circuits:
        local_to_global_map = {i: q_idx for i, q_idx in enumerate(qubits_list)}
        for inst, qargs, cargs in cir.data:
            global_qargs = [qc_rebuilt_original_qubits.qubits[local_to_global_map[cir.find_bit(q).index]] for q in qargs]
            qc_rebuilt_original_qubits.append(inst, global_qargs, cargs)

    print("\nRecomposed circuit (before interface SWAP and duplication):")
    print(qc_rebuilt_original_qubits.draw(output="text"))
    fid_rebuilt = compute_fidelity(qc_rebuilt_original_qubits, U_orig)
    print(f"Recomposed fidelity ↔ original: {fid_rebuilt:.5f}")

    # Reinjection of original inter-block gates
    for inst, qargs, cargs in original_interblock_gates:
        global_qargs = [qc_rebuilt_original_qubits.qubits[qc.find_bit(q).index] for q in qargs]
        qc_rebuilt_original_qubits.append(inst, global_qargs, cargs)

    print("📎 Inter-block gates reinjected into the recomposed circuit.")
    fid_rebuilt1 = compute_fidelity(qc_rebuilt_original_qubits, U_orig)
    print(f"Recomposed fidelity (with inter-blocks) ↔ original: {fid_rebuilt1:.5f}")
    print("\nRecomposed circuit with inter-block gates:")
    print(qc_rebuilt_original_qubits.draw(output="text"))

    # 4. Inter-block injection
    if injection_method == "sa":
        qc_inj, kept = sa_injection(qc_rebuilt_original_qubits, original_blocks, fid_threshold=fid_threshold,
                                    n_iters=sa_iters, seed=sa_seed)
    elif injection_method == "stochastic":
        qc_inj, kept = stochastic_injection(qc_rebuilt_original_qubits, original_blocks, fid_threshold=fid_threshold)
    else:
        raise ValueError('injection_method must be "sa" or "stochastic".')

    print("\nCircuit after inter-block injection:")
    print(qc_inj.draw(output="text"))
    print(f"# inter-block gates kept: {len(kept)}")
    fid_inj = compute_fidelity(qc_inj, U_orig)
    print(f"Fidelity after inter-block injection ↔ original: {fid_inj:.5f}")

    # 4.1 Injection driven by fidelity (greedy)
    qc_i, kept1 = fidelity_driven_injection(base_qc=qc_rebuilt_original_qubits, target_qc=qc_orig,
                                            blocks=original_blocks, max_trials=300, fid_threshold=0.9999)
    print("\nCircuit after inter-block injection with NSGA2 (greedy):")
    print(qc_i.draw(output="text"))
    qc_i.draw('mpl', filename=f"final_optimized_circuitwithdriveninject.png", style='mpl', fold=1)
    print(f"# inter-block gates kept: {len(kept1)}")
    fid_i = compute_fidelity(qc_i, U_orig)
    print(f"Fidelity after inter-block injection ↔ original: {fid_i:.5f}")

    # 5. Final compression
    if fid_i > fid_inj:
        qc_opt = compress_custom(qiskit_opt_pass(qc_i))
    else:
        qc_opt = compress_custom(qiskit_opt_pass(qc_inj))

    print("\nFinal optimized circuit:")
    print(qc_opt.draw(output="text"))
    qc_opt.draw('mpl', filename=f"final_optimized_circuit.png", style='mpl', fold=1)
    cost_final = compute_gate_cost(qc_opt)
    print(f"💰 Cost of the final optimized circuit (Lee et al. 2006): {cost_final}")

    fid_final = compute_fidelity(qc_opt, U_orig)
    depth_before = qc_orig.depth()
    depth_after = qc_opt.depth()

    print("\n===== Final Summary =====")
    print("🎯 Final global fidelity:", fid_final)
    print("📏 Depth (original):", depth_before)
    print("📏 Depth (optimized):", depth_after)
    print("Total qubits (original):", qc_orig.num_qubits)
    print("Total qubits (final):", qc_opt.num_qubits)
    print(f"💰 Cost of the final circuit:", cost_final)

    meta = {
        "blocks": original_blocks,
        "kept_injections": kept,
        "depth_before": depth_before,
        "depth_after": depth_after,
        "fidelity_final": fid_final,
        "original_num_qubits": qc_orig.num_qubits,
        "final_num_qubits": qc_opt.num_qubits,
        "highly_interactive_qubits_identified": highly_interactive_qubits,
        "cost_before": cost_orig,
        "cost_after": cost_final
    }
    return qc_opt, meta

# Qiskit 2.0 compatible
from qiskit import QuantumCircuit
import numpy as np

def rzz_via_cx(qc: QuantumCircuit, i: int, j: int, theta: float):
    # RZZ(theta) = CX(i,j); RZ(theta) on j; CX(i,j)
    qc.cx(i, j)
    qc.rz(theta, j)
    qc.cx(i, j)

def build_sparse_8q_qaoa_circuit(gamma: float = 0.7, beta: float = 0.4) -> QuantumCircuit:
    n = 8
    qc = QuantumCircuit(n, name="QAOA_8q_sparse")

    # Ring in block A (0..3) and block B (4..7)
    ring_A = [(0,1), (1,2), (2,3), (3,0)]
    ring_B = [(4,5), (5,6), (6,7), (7,4)]
    bridge = [(3,4)]  # low connectivity between blocks

    edges = ring_A + ring_B + bridge

    # 1) Initial superposition
    for q in range(n):
        qc.h(q)

    # 2) Cost layer (Max-Cut): RZZ on the edges
    for (u, v) in edges:
        rzz_via_cx(qc, u, v, gamma)

    # 3) Mixing layer
    for q in range(n):
        qc.rx(2*beta, q)  # RX(2*beta) common in QAOA

    qc.barrier()
    return qc, edges


# =============================
# Example main
# =============================
if __name__ == "__main__":
    from qiskit import QuantumCircuit
    # Circuit on 10 qubits
    qc = QuantumCircuit(10)
    qc.ry(4.66, 0)
    qc.ry(1.69, 1)
    qc.cz(2, 3)
    qc.rz(4.56, 3)
    qc.x(3)
    qc.rz(6.81, 4)
    qc.y(4)
    qc.h(4)
    qc.rz(3.27, 5)
    qc.x(5)
    qc.y(5)
    qc.z(5)
    qc.rz(0.91, 6)
    qc.x(6)
    qc.y(6)
    qc.z(6)
    qc.rz(4.62, 7)
    qc.y(7)
    qc.rz(3.93, 8)
    qc.x(8)
    qc.y(8)
    qc.z(8)
    qc.rzz(6.63, 9, 0)  # RZZ between q9 and q0
    qc.cx(0, 2)
    qc.cx(1, 2)
    qc.z(2)
    qc.cx(3, 6)
    qc.cx(4, 7)
    qc.z(6)
    qc.cx(6, 9)
    qc.cx(7, 8)
    qc.cx(8, 9)



    

    qc_final, info = optimise_circuit_pipeline(
        qc,
        max_block_size=6,
        k_interface=1,
        injection_method="stochastic",
        fid_threshold=0.9999,
        sa_iters=3000,
        sa_seed=0,
        qubit_duplication_threshold=0.6,
    )

    print("\n===== Summary (main) =====")
    for k, v in info.items():
        if k == "blocks":
            print("Blocks:", v)
        else:
            print(f"{k.replace('_', ' ').title()} : {v}")



Circuit original :
     ┌──────────┐                                                          
q_0: ┤ Ry(4.66) ├─────────────■───────────■────────────────────────────────
     ├──────────┤             │           │                                
q_1: ┤ Ry(1.69) ├─────────────┼───────────┼────■───────────────────────────
     └──────────┘             │         ┌─┴─┐┌─┴─┐┌───┐                    
q_2: ─────■───────────────────┼─────────┤ X ├┤ X ├┤ Z ├────────────────────
          │      ┌──────────┐ │         ├───┤└───┘└───┘                    
q_3: ─────■──────┤ Rz(4.56) ├─┼─────────┤ X ├────────────■─────────────────
     ┌──────────┐└──┬───┬───┘ │         ├───┤            │                 
q_4: ┤ Rz(6.81) ├───┤ Y ├─────┼─────────┤ H ├───────■────┼─────────────────
     ├──────────┤   ├───┤     │         ├───┤┌───┐  │    │                 
q_5: ┤ Rz(3.27) ├───┤ X ├─────┼─────────┤ Y ├┤ Z ├──┼────┼─────────────────
     ├──────────┤   ├───┤     │         ├───┤├───┤  │  ┌─┴─┐┌───┐   

C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:477: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  return sum(cost_table.get(inst.name.lower(), 1) for inst, _, _ in qc.data)
C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:129: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in qc.data:
C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:96: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, cargs in qc

💰 Coût du circuit original (Lee et al. 2006) : 65

📌 Partitionnement du circuit initial…
Qubits par bloc (initial) : ({0, 1, 2, 3, 5}, {4, 6, 7, 8, 9})
📎 2 portes inter-blocs extraites pour réinjection plus tard.
🧭 Affichage du graphe d’interaction… avant duplication


C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:232: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, _ in qc.data:
C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:1123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(); save_plot("interaction_graph"); plt.close()


💡 Aucune duplication de qubit nécessaire ou identifiée.
🧭 Affichage du graphe d’interaction…


...

Fidélité recomposé ↔ original: 0.16666
📎 Portes inter-blocs réinjectées dans le circuit recomposé.
Fidélité recomposé (avec inter-blocs) ↔ original: 0.15616

Circuit recomposé avec portes interblocs :
                      ┌───┐                                                 »
q_0: ─■───────────────┤ H ├────────────────────────────────────■────────────»
      │               ├───┤          ┌───┐     ┌─────────────┐ │ZZ(-1.4615) »
q_1: ─┼───────────────┤ H ├──────────┤ Z ├─────┤ Ry(-3.0244) ├─■────────────»
      │ZZ(3.0838)     └───┘     ┌────┴───┴────┐└┬────────────┤              »
q_2: ─■────────────■────────────┤ Ry(-1.4631) ├─┤ Rz(1.4826) ├──────────────»
                   │ZZ(-1.4566) └────┬───┬────┘ └───┬───┬────┘              »
q_3: ──────────────■─────────────────┤ Z ├──────────┤ Y ├───────────────────»
                      ┌───┐          ├───┤          ├───┤         ┌───┐     »
q_4: ─────────────────┤ Y ├──────────┤ H ├──────────┤ H ├─────────┤ H ├─────»
                   

C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:426: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  g1, q1, _ = c.data[i]; g2, q2, _ = c.data[i + 1]
C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:431: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  g, q, _ = c.data[-1]; new.append(g, q)
C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:438: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  g, q, _ = c.data[i]
C:\Users\Thinkpad\AppData


Circuit après injection inter-blocs :
                      ┌───┐                                                 »
q_0: ─■───────────────┤ H ├────────────────────────────────────■────────────»
      │               ├───┤          ┌───┐     ┌─────────────┐ │ZZ(-1.4615) »
q_1: ─┼───────────────┤ H ├──────────┤ Z ├─────┤ Ry(-3.0244) ├─■────────────»
      │ZZ(3.0838)     └───┘     ┌────┴───┴────┐└┬────────────┤              »
q_2: ─■────────────■────────────┤ Ry(-1.4631) ├─┤ Rz(1.4826) ├──────────────»
                   │ZZ(-1.4566) └────┬───┬────┘ └───┬───┬────┘              »
q_3: ──────────────■─────────────────┤ Z ├──────────┤ Y ├───────────────────»
                      ┌───┐          ├───┤          ├───┤         ┌───┐     »
q_4: ─────────────────┤ Y ├──────────┤ H ├──────────┤ H ├─────────┤ H ├─────»
                      ├───┤          ├───┤          └───┘         └───┘     »
q_5: ─────────────────┤ X ├──────────┤ Y ├──────────────────────────────────»
                      ├──

C:\Users\Thinkpad\AppData\Local\Temp\ipykernel_3084\2763840989.py:477: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  return sum(cost_table.get(inst.name.lower(), 1) for inst, _, _ in qc.data)


💰 Coût du circuit optimisé final (Lee et al. 2006) : 54

===== Résumé Final =====
🎯 Fidélité globale finale : 0.1641600053049518
📏 Profondeur (original) : 8
📏 Profondeur (optimisé) : 6
Total qubits (original): 10
Total qubits (final): 10
💰 Coût du circuit final: 54

===== Résumé (main) =====
Blocks : [{0, 1, 2, 3, 5}, {4, 6, 7, 8, 9}]
Kept Injections : []
Depth Before : 8
Depth After : 6
Fidelity Final : 0.1641600053049518
Original Num Qubits : 10
Final Num Qubits : 10
Highly Interactive Qubits Identified : {}
Cost Before : 65
Cost After : 54
